# Explainability & Interpretability — CNN-LSTM ECG Forecaster

Techniques used:
1. **Gradient × Input** — which input features drive each prediction
2. **Temporal importance** — which past time-steps matter most
3. **Lead importance** — which ECG leads are most influential
4. **Occlusion sensitivity** — behaviour when individual leads are zeroed out
5. **Error decomposition** — where and why the model makes mistakes

## 1. Setup and Imports

In [ ]:
import os
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from scipy.ndimage import gaussian_filter1d
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')

## 2. Paths and Configuration

In [ ]:
SAVE_DIR  = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'reports', 'checkpoints')
FIG_DIR   = os.path.join('..', 'reports', 'figures', 'explainability')
os.makedirs(FIG_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Constants — must match the training notebook exactly
FS         = 100
INPUT_LEN  = 500
HORIZON    = 100
N_LEADS    = 12
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

COLORS = {
    'bg':      '#ffffff',
    'grid':    '#e0e0e0',
    'text':    '#2c3e50',
    'accent1': '#3498db',
    'accent2': '#e74c3c',
    'accent3': '#2ecc71',
    'accent4': '#f39c12',
}

plt.rcParams.update({
    'figure.facecolor': COLORS['bg'],
    'axes.facecolor':   COLORS['bg'],
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'figure.dpi':       120,
    'savefig.dpi':      150,
})

def save_fig(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {name}")

print(f"Device: {DEVICE}")
print(f"Input: {INPUT_LEN/FS:.1f}s  |  Horizon: {HORIZON/FS:.1f}s")

## 3. Load Data

In [ ]:
X_test = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))

with open(os.path.join(SAVE_DIR, 'config.pkl'), 'rb') as f:
    config = pickle.load(f)

n_samples_explain = min(50, len(X_test))
X_explain = X_test[:n_samples_explain]
y_explain = y_test[:n_samples_explain]

X_explain_torch = torch.tensor(X_explain, dtype=torch.float32).to(DEVICE)
y_explain_torch = torch.tensor(y_explain, dtype=torch.float32).to(DEVICE)

print(f"X_explain: {X_explain.shape}")
print(f"y_explain: {y_explain.shape}")

## 4. CNN-LSTM Model Definition

Must match the architecture saved during training exactly.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, pool=True):
        super().__init__()
        ops = [
            nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=k // 2, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
        ]
        if pool:
            ops.append(nn.MaxPool1d(2))
        self.net = nn.Sequential(*ops)

    def forward(self, x):
        return self.net(x)


class CNNLSTMForecaster(nn.Module):
    def __init__(self, n_leads=12, horizon=100, dropout=0.2,
                 lstm_hidden=128, bidirectional=True):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads
        self.D       = 2 if bidirectional else 1

        self.cnn = nn.Sequential(
            ConvBlock(n_leads, 32,  k=7, pool=True),
            ConvBlock(32,      64,  k=5, pool=True),
            ConvBlock(64,      128, k=3, pool=True),
            ConvBlock(128,     128, k=3, pool=False),
        )
        self.lstm = nn.LSTM(
            input_size=128, hidden_size=lstm_hidden, num_layers=2,
            batch_first=True, dropout=dropout, bidirectional=bidirectional,
        )
        self.attn = nn.Sequential(
            nn.Linear(lstm_hidden * self.D, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        self.decoder = nn.Sequential(
            nn.LayerNorm(lstm_hidden * self.D),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * self.D, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, horizon * n_leads),
        )

    def forward(self, x):
        f      = self.cnn(x).permute(0, 2, 1)       # (B, T', 128)
        enc, _ = self.lstm(f)                         # (B, T', 2*hidden)
        w      = torch.softmax(self.attn(enc), dim=1)
        ctx    = (w * enc).sum(dim=1)                 # (B, 2*hidden)
        out    = self.decoder(ctx)
        return out.view(-1, self.horizon, self.n_leads)

## 5. Load Trained Model

In [ ]:
model = CNNLSTMForecaster(n_leads=N_LEADS, horizon=HORIZON, dropout=0.2).to(DEVICE)

model_path = os.path.join(MODEL_DIR, 'CNN-LSTM_final.pt')
if os.path.exists(model_path):
    # weights_only=True required for PyTorch >= 2.4
    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=True)
    state_dict = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    model.load_state_dict(state_dict)
    print("✅ Model loaded successfully")
else:
    print("⚠️  No checkpoint found — using untrained model (results will be random)")

model.eval()
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Gradient × Input Attribution

Computes the element-wise product of the input with its gradient w.r.t. the MSE loss.
Positive values = features that pushed prediction toward the true value;
negative values = features that pulled it away.

In [ ]:
def compute_gradient_attribution(model, X_batch, y_batch):
    """
    Returns
    -------
    attribution_np : ndarray  (N, T, C)
    predictions_np : ndarray  (N, horizon, C)
    """
    # model must be in eval mode with grad enabled for the input only
    model.eval()
    X_in   = X_batch.clone().detach().requires_grad_(True)
    y_pred = model(X_in)
    loss   = F.mse_loss(y_pred, y_batch)
    loss.backward()
    attribution = (X_in.grad * X_in.detach()).detach()
    return attribution.cpu().numpy(), y_pred.detach().cpu().numpy()


print("Computing gradient-based attributions …")
attributions_np, predictions_np = compute_gradient_attribution(
    model, X_explain_torch, y_explain_torch)

print(f"Attributions : {attributions_np.shape}")
print(f"Predictions  : {predictions_np.shape}")

## 7. Attribution Heatmaps

In [ ]:
n_viz = 3
fig   = plt.figure(figsize=(16, 3.5 * n_viz))

for idx in range(n_viz):
    attr_sample = attributions_np[idx]

    # -- Attribution map --
    ax1  = plt.subplot(n_viz, 2, 2 * idx + 1)
    vmax = max(abs(attr_sample.min()), abs(attr_sample.max()))  # symmetric around 0
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im1  = ax1.imshow(attr_sample.T, aspect='auto', cmap='RdBu_r', norm=norm)
    ax1.set_yticks(range(N_LEADS))
    ax1.set_yticklabels(LEAD_NAMES, fontsize=8)
    ax1.set_xlabel('Time (samples)')
    ax1.set_ylabel('Lead')
    ax1.set_title(f'Sample {idx+1}: Gradient×Input Attribution', fontweight='bold')
    plt.colorbar(im1, ax=ax1, label='Attribution')

    # -- Prediction error map --
    ax2        = plt.subplot(n_viz, 2, 2 * idx + 2)
    pred_error = predictions_np[idx] - y_explain[idx]
    emax       = max(abs(pred_error.min()), abs(pred_error.max()))
    norm2      = TwoSlopeNorm(vmin=-emax, vcenter=0, vmax=emax)
    im2        = ax2.imshow(pred_error.T, aspect='auto', cmap='RdBu_r', norm2=norm2 if emax > 0 else None)
    ax2.set_yticks(range(N_LEADS))
    ax2.set_yticklabels(LEAD_NAMES, fontsize=8)
    ax2.set_xlabel('Time (samples)')
    ax2.set_ylabel('Lead')
    ax2.set_title(f'Sample {idx+1}: Prediction Error', fontweight='bold')
    plt.colorbar(im2, ax=ax2, label='Error')

plt.suptitle('Gradient-Based Attribution & Prediction Error', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('01_gradient_attribution_heatmaps.png')

## 8. Temporal and Lead Importance

In [ ]:
time_importance          = np.abs(attributions_np).mean(axis=2)  # (N, T)
lead_importance          = np.abs(attributions_np).mean(axis=1)  # (N, C)
time_importance_smoothed = np.array([gaussian_filter1d(t, sigma=3) for t in time_importance])
mean_lead_importance     = lead_importance.mean(axis=0)
std_lead_importance      = lead_importance.std(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Temporal importance
for i in range(min(10, len(time_importance_smoothed))):
    axes[0].plot(time_importance_smoothed[i], label=f'Sample {i+1}', alpha=0.7, linewidth=1.5)
axes[0].axvline(x=INPUT_LEN - 50, color=COLORS['accent2'], linestyle='--',
                linewidth=2, label='Recent history start')
axes[0].set_xlabel('Time Index (samples)')
axes[0].set_ylabel('Mean |Attribution|')
axes[0].set_title('Temporal Importance: Which Past Time Steps Matter Most?', fontweight='bold')
axes[0].legend(fontsize=8, loc='upper right')

# Lead importance
colors_lead = plt.cm.Set3(np.linspace(0, 1, N_LEADS))
axes[1].bar(LEAD_NAMES, mean_lead_importance, yerr=std_lead_importance,
            capsize=4, color=colors_lead, edgecolor=COLORS['text'], linewidth=1.5)
axes[1].set_ylabel('Mean |Attribution|')
axes[1].set_title('Lead Importance: Which Leads Drive Predictions?', fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
save_fig('02_temporal_lead_importance.png')

top_leads = np.argsort(mean_lead_importance)[-3:][::-1]
print("Top 3 important leads (Gradient×Input):")
for rank, li in enumerate(top_leads, 1):
    print(f"  {rank}. {LEAD_NAMES[li]} ({mean_lead_importance[li]:.4f})")

## 9. Sample Predictions vs Ground Truth

In [ ]:
n_show        = 3
leads_to_show = [0, 1, 6, 7, 10, 11]
fig           = plt.figure(figsize=(15, 4 * n_show))

for sample_idx in range(n_show):
    for subplot_idx, lead_idx in enumerate(leads_to_show):
        ax      = plt.subplot(n_show, 6, sample_idx * 6 + subplot_idx + 1)
        x_input = X_explain[sample_idx, :, lead_idx]
        t_input = np.arange(len(x_input))
        y_true  = y_explain[sample_idx, :, lead_idx]
        t_true  = np.arange(INPUT_LEN, INPUT_LEN + len(y_true))
        y_pred  = predictions_np[sample_idx, :, lead_idx]

        ax.plot(t_input, x_input, color=COLORS['accent3'], label='Input',        linewidth=1.5)
        ax.plot(t_true,  y_true,  color=COLORS['accent1'], label='Ground Truth', linewidth=2)
        ax.plot(t_true,  y_pred,  color=COLORS['accent2'], label='Prediction',   linewidth=2, linestyle='--')
        ax.axvspan(INPUT_LEN, INPUT_LEN + HORIZON, alpha=0.1, color=COLORS['accent1'])
        ax.set_title(f'Lead {LEAD_NAMES[lead_idx]}', fontsize=9)
        ax.set_xlabel('Sample')
        ax.set_ylabel('ECG (norm.)')
        ax.grid(True, alpha=0.2)
        if subplot_idx == 0:
            ax.legend(fontsize=7, loc='upper left')

plt.suptitle('Sample Predictions vs Ground Truth', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('03_sample_predictions.png')

## 10. Occlusion Sensitivity

Each lead is zeroed out in turn; the increase in MSE loss measures how much the model depends on that lead.

In [ ]:
def compute_occlusion_sensitivity(model, X_batch, y_batch, n_leads=12):
    model.eval()
    with torch.no_grad():
        baseline_loss = F.mse_loss(model(X_batch), y_batch).item()

    impacts = []
    for lead_idx in range(n_leads):
        X_occ = X_batch.clone()
        X_occ[:, :, lead_idx] = 0.0
        with torch.no_grad():
            occluded_loss = F.mse_loss(model(X_occ), y_batch).item()
        impacts.append(occluded_loss - baseline_loss)
    return np.array(impacts)


print("Computing occlusion sensitivities …")
batch_size              = 10
occlusion_sensitivities = []

for i in range(0, len(X_explain_torch), batch_size):
    end  = min(i + batch_size, len(X_explain_torch))
    sens = compute_occlusion_sensitivity(
        model, X_explain_torch[i:end], y_explain_torch[i:end])
    occlusion_sensitivities.append(sens)

occlusion_sensitivities = np.array(occlusion_sensitivities)   # (n_batches, N_LEADS)
mean_occlusion          = occlusion_sensitivities.mean(axis=0)
std_occlusion           = occlusion_sensitivities.std(axis=0)
print("Done.")

fig, axes   = plt.subplots(1, 2, figsize=(14, 5))
colors_occ  = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, N_LEADS))

axes[0].bar(LEAD_NAMES, mean_occlusion, yerr=std_occlusion, capsize=4,
            color=colors_occ, edgecolor=COLORS['text'], linewidth=1.5)
axes[0].set_ylabel('Loss Increase (MSE)')
axes[0].set_title('Occlusion Sensitivity: Lead Importance by Ablation', fontweight='bold')
axes[0].grid(True, axis='y', alpha=0.3)

im = axes[1].imshow(occlusion_sensitivities[:15], aspect='auto', cmap='YlOrRd')
axes[1].set_xlabel('Lead')
axes[1].set_ylabel('Sample index')
axes[1].set_xticks(range(N_LEADS))
axes[1].set_xticklabels(LEAD_NAMES, fontsize=9)
axes[1].set_title('Per-Sample Occlusion Sensitivity', fontweight='bold')
plt.colorbar(im, ax=axes[1], label='Loss Increase')

plt.tight_layout()
save_fig('04_occlusion_sensitivity.png')

## 11. Error Decomposition

In [ ]:
pred_errors     = predictions_np - y_explain
rmse_per_sample = np.sqrt(np.mean(pred_errors ** 2, axis=(1, 2)))
mae_per_sample  = np.mean(np.abs(pred_errors),       axis=(1, 2))
error_by_lead   = np.abs(pred_errors).mean(axis=(0, 1))   # (N_LEADS,)
error_by_time   = np.abs(pred_errors).mean(axis=(0, 2))   # (HORIZON,)
rmse_by_lead    = np.sqrt(np.mean(pred_errors ** 2, axis=(0, 1)))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Error by lead
axes[0, 0].bar(LEAD_NAMES, error_by_lead,
               color=plt.cm.viridis(np.linspace(0, 1, N_LEADS)))
axes[0, 0].set_ylabel('Mean |Error|')
axes[0, 0].set_title('Prediction Error by Lead', fontweight='bold')
axes[0, 0].grid(True, axis='y', alpha=0.3)

# Error over forecast horizon
axes[0, 1].fill_between(range(HORIZON), error_by_time, alpha=0.3, color=COLORS['accent1'])
axes[0, 1].plot(error_by_time, color=COLORS['accent1'], linewidth=2)
axes[0, 1].set_xlabel('Forecast Time Step')
axes[0, 1].set_ylabel('Mean |Error|')
axes[0, 1].set_title('Prediction Error Over Forecast Horizon', fontweight='bold')

# Importance vs error scatter
axes[1, 0].scatter(mean_lead_importance, rmse_by_lead, s=100, alpha=0.6,
                   color=COLORS['accent1'], edgecolor=COLORS['text'], linewidth=1.5)
for i, lead in enumerate(LEAD_NAMES):
    axes[1, 0].annotate(lead, (mean_lead_importance[i], rmse_by_lead[i]), fontsize=8)
axes[1, 0].set_xlabel('Attribution Importance')
axes[1, 0].set_ylabel('RMSE')
axes[1, 0].set_title('Lead Importance vs Prediction Error', fontweight='bold')

# Error heatmap
im = axes[1, 1].imshow(np.abs(pred_errors).mean(axis=0).T, aspect='auto', cmap='YlOrRd')
axes[1, 1].set_xlabel('Forecast Time Step')
axes[1, 1].set_ylabel('Lead')
axes[1, 1].set_yticks(range(N_LEADS))
axes[1, 1].set_yticklabels(LEAD_NAMES, fontsize=9)
axes[1, 1].set_title('Error Heatmap: Lead × Time', fontweight='bold')
plt.colorbar(im, ax=axes[1, 1], label='|Error|')

plt.tight_layout()
save_fig('05_error_analysis.png')

## 12. Explainability Summary Dashboard

In [ ]:
correlation = np.corrcoef(mean_lead_importance, mean_occlusion)[0, 1]

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 3, figure=fig)

# Attribution distribution
ax = fig.add_subplot(gs[0, 0])
ax.hist(attributions_np.flatten(), bins=50,
        color=COLORS['accent1'], edgecolor=COLORS['text'], alpha=0.7)
ax.set_xlabel('Gradient x Input')
ax.set_ylabel('Frequency')
ax.set_title('Attribution Distribution', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Gradient-based importance
ax = fig.add_subplot(gs[0, 1])
colors_grad = plt.cm.Set2(np.linspace(0, 1, N_LEADS))
ax.barh(LEAD_NAMES[::-1], mean_lead_importance[::-1], color=colors_grad[::-1])
ax.set_xlabel('Mean |Attribution|')
ax.set_title('Gradient-Based Importance', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Occlusion-based importance
ax = fig.add_subplot(gs[0, 2])
ax.barh(LEAD_NAMES[::-1], mean_occlusion[::-1],
        color=plt.cm.Oranges(np.linspace(0.4, 0.9, N_LEADS))[::-1])
ax.set_xlabel('Loss Increase')
ax.set_title('Occlusion-Based Importance', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# RMSE by lead
ax = fig.add_subplot(gs[1, 0])
ax.bar(LEAD_NAMES, rmse_by_lead,
       color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, N_LEADS)))
ax.set_ylabel('RMSE')
ax.set_title('Prediction Accuracy by Lead', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Error over horizon
ax = fig.add_subplot(gs[1, 1:])
ax.fill_between(range(HORIZON), error_by_time, alpha=0.3, color=COLORS['accent2'])
ax.plot(error_by_time, color=COLORS['accent2'], linewidth=2)
ax.set_xlabel('Forecast Step')
ax.set_ylabel('Mean |Error|')
ax.set_title('Error Over Forecast Horizon', fontweight='bold')

# Method agreement scatter
ax = fig.add_subplot(gs[2, 0])
ax.scatter(mean_lead_importance, mean_occlusion, s=150, alpha=0.7,
           color=COLORS['accent3'], edgecolor=COLORS['text'], linewidth=1.5)
for i, lead in enumerate(LEAD_NAMES):
    ax.annotate(lead, (mean_lead_importance[i], mean_occlusion[i]), fontsize=8)
ax.set_xlabel('Gradient-Based Importance')
ax.set_ylabel('Occlusion Importance')
ax.set_title('Method Agreement  r=' + f'{correlation:.3f}', fontweight='bold')

# Key insights text box
ax = fig.add_subplot(gs[2, 1:])
ax.axis('off')
top3 = ', '.join(LEAD_NAMES[li] for li in top_leads)
insights_lines = [
    'KEY INSIGHTS - CNN-LSTM EXPLAINABILITY',
    '',
    'Model Performance:',
    f'  Mean RMSE : {rmse_per_sample.mean():.4f}',
    f'  Mean MAE  : {mae_per_sample.mean():.4f}',
    '',
    'Most Important Leads (Gradient):',
    f'  1. {LEAD_NAMES[top_leads[0]]}  ({mean_lead_importance[top_leads[0]]:.4f})',
    f'  2. {LEAD_NAMES[top_leads[1]]}  ({mean_lead_importance[top_leads[1]]:.4f})',
    f'  3. {LEAD_NAMES[top_leads[2]]}  ({mean_lead_importance[top_leads[2]]:.4f})',
    '',
    'Temporal Pattern:',
    '  Recent samples carry higher importance',
    '  Error grows toward end of horizon',
    '',
    f'Method Agreement: r = {correlation:.3f}',
]
insights = '\n'.join(insights_lines)
ax.text(0.05, 0.95, insights, transform=ax.transAxes, fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle('Explainability Summary Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('06_explainability_summary.png')

print("\n Explainability Analysis Complete")
print(f"Figures saved to: {os.path.abspath(FIG_DIR)}")